In [18]:
from dotenv import load_dotenv
load_dotenv()

True

In [19]:
import pandas as pd

In [20]:
df = pd.read_csv('./data/North Korean Provocations.csv')
df

,Date,Type,Event,Description,Resources
0,1958-02-16,Other Provocation,Hijacking of South Korean Plane,North Korean agents hijacked a South Korean pl...,https://goo.gl/Xn9Wr4 The US Military Experien...
1,1959-06-15,Other Provocation,NK Fighter Jets Attack U.S. Navy Plane,Two North Korean jet fighters (MiG) attacked a...,https://goo.gl/UFh93r The US Military Experien...
2,1960-07-30,Other Provocation,Exchange of Fire and Sinking of NK Vessel,A South Korean destroyer sinks a North Korean ...,https://goo.gl/NEgR7x The US Military Experien...
3,1962-11-20,Other Provocation,Exchange of Fire at DMZ,NK troops attacked a UN observation post 7 mil...,http://www.cfr.org/content/publications/attach...
4,1963-07-29,Other Provocation,Exchange of Fire at DMZ,A group of at least seven North Korean soldier...,https://goo.gl/8PXjzE; The US Military Experie...
...,...,...,...,...,...
477,2026-01-27,Missile Provocation,Short-range Ballistic Missile Launch,South Korea’s Joint Chiefs of Staff (JCS) repo...,https://koreajoongangdaily.joins.com/news/2026...
478,2026-01-28,Missile Provocation,Multiple Lanuch Rocket System (MLRS) Test,North Korea's Kim Jong-un oversaw a test launc...,https://www.nknews.org/2026/01/kim-jong-un-say...
479,2026-03-04,Missile Provocation,Strategic cruise missile,Kim Jong-un oversaw the test launch of fleet-t...,https://biz.chosun.com/en/en-policy/2026/03/05...
480,2026-03-11,Missile Provocation,Strategic cruise missile,North Korea's state media KCNA reported that K...,https://www.koreatimes.co.kr/amp/foreignaffair...


In [21]:
df = df.drop(columns=['Resources'])
df

,Date,Type,Event,Description
0,1958-02-16,Other Provocation,Hijacking of South Korean Plane,North Korean agents hijacked a South Korean pl...
1,1959-06-15,Other Provocation,NK Fighter Jets Attack U.S. Navy Plane,Two North Korean jet fighters (MiG) attacked a...
2,1960-07-30,Other Provocation,Exchange of Fire and Sinking of NK Vessel,A South Korean destroyer sinks a North Korean ...
3,1962-11-20,Other Provocation,Exchange of Fire at DMZ,NK troops attacked a UN observation post 7 mil...
4,1963-07-29,Other Provocation,Exchange of Fire at DMZ,A group of at least seven North Korean soldier...
...,...,...,...,...
477,2026-01-27,Missile Provocation,Short-range Ballistic Missile Launch,South Korea’s Joint Chiefs of Staff (JCS) repo...
478,2026-01-28,Missile Provocation,Multiple Lanuch Rocket System (MLRS) Test,North Korea's Kim Jong-un oversaw a test launc...
479,2026-03-04,Missile Provocation,Strategic cruise missile,Kim Jong-un oversaw the test launch of fleet-t...
480,2026-03-11,Missile Provocation,Strategic cruise missile,North Korea's state media KCNA reported that K...


- langchain에서는 list 형식보다 document 형식이 더 선호.
- 나중에 metadata를 추가할 때도 document 형식이 더 유용할 수 있음.

### RAG 사용

In [22]:
from langchain_core.documents import Document

docs = []
for idx,row in df.iterrows():
    content= f"{row['Date']} {row['Event']} {row['Description']}"

    doc = Document(
        page_content = content,
        metadata = {
            'date': str(row['Date']),
            'Type' : str(row['Type']),
            'event': str(row['Event']),
            'description': str(row['Description'])
            }
    )
    docs.append(doc)

- Embedding 모델 호출

In [23]:
from langchain_openai.embeddings import OpenAIEmbeddings # langchai 전용?

embedding_model = OpenAIEmbeddings(model='text-embedding-3-small')

#### Store
- chroma

In [24]:
from langchain_chroma.vectorstores import Chroma

vector_store = Chroma.from_documents(
    documents = docs, 
    embedding = embedding_model,
    collection_name = 'north_korea_events'
)
vector_store

In [25]:
vector_store.similarity_search_with_score('북한이 2020년에 한 도발은?')

[(Document(id='5955072d-77ea-4d0f-927f-20c6e765ecc1', metadata={'event': 'Short-range Ballistic Missile Launch', 'Type': 'Missile Provocation', 'date': '2022-08-17', 'description': 'North Korea fired two cruise missiles from Onchon, South Pyongan province into the sea between Korea and China.'}, page_content='2022-08-17 Short-range Ballistic Missile Launch North Korea fired two cruise missiles from Onchon, South Pyongan province into the sea between Korea and China.'),
  1.0262401103973389),
 (Document(id='28d7780c-4f1a-4044-b44c-41d848ea675e', metadata={'event': 'Short-range Missile Launch', 'date': '2020-03-21', 'description': 'On March 21, 2020 at 6:45 a.m. and 6:50 p.m., North Korea launched two projectiles from the county of Sonchon in North Pyongan Province towards the sea between Korea and Japan. The projectiles traveled a distance of 410km (255 miles) at a maximum altitude of 50km (31 miles) and closely resemble the KN-24 short-range ballistic missile system.', 'Type': 'Missile

In [26]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """
너는 북한의 과거 도발 사례를 분석하는 안보 데이터 해설 전문가다.

역할:
- 사용자가 입력한 현재 사건이나 질문을 바탕으로,
  제공된 과거 사례(context)를 참고하여 유사 사례를 비교·설명한다.
- 반드시 예측처럼 단정하지 말고, 과거 데이터 기반 비교 설명만 수행한다.
- "가능성이 높다/낮다" 같은 표현을 쓰더라도, 그것은 예측이 아니라
  과거 사례에서 관찰된 패턴 수준의 참고 설명임을 분명히 밝혀라.

답변 원칙:
1. 제공된 context 안의 정보만 근거로 사용하라.
2. context에 없는 내용을 지어내지 마라.
3. 정보가 부족하면 "제공된 과거 사례만으로는 충분히 판단하기 어렵다"고 말하라.
4. 한국어로 답변하라.
5. 사용자의 질문이 현재 사건의 위험성이나 한국에 미칠 영향에 대한 것이라면,
   과거 유사 사례에서 나타난 영향 양상을 중심으로 설명하라.
6. 군사 작전 지시, 실시간 대응 명령, 확정적 예측은 하지 마라.

답변 형식:
- 사건 해석:
- 유사한 과거 사례 비교:
- 과거 사례에서 나타난 후속 양상:
- 한국에 미칠 수 있었던 영향(과거 사례 기준):
- 한계점:

context:
{context}
            """
        ),
        (
            "human",
            """
사용자 질문:
{question}
            """
        )
    ]
)

In [27]:
#  impor retriever
retriever = vector_store.as_retriever( 
  search_type='similarity',
  search_kwargs={'k': 3}
)

retriever.batch(['북한이 드론을 이용한 도발을 했을 때 한국에 미친 영향은?'])

[[Document(id='ebe0252e-eb12-4d81-ac0c-9be562151161', metadata={'date': '1971-06-01', 'event': 'Infiltration and Incursion', 'Type': 'Other Provocation', 'description': 'A combined South Korean air force and navy operation sinks a 70-ton North Korean infiltration speedboat off the western coast south of the MDL. It is believed that the speedboat was carrying around 17 North Korean agents.'}, page_content='1971-06-01 Infiltration and Incursion A combined South Korean air force and navy operation sinks a 70-ton North Korean infiltration speedboat off the western coast south of the MDL. It is believed that the speedboat was carrying around 17 North Korean agents.'),
  Document(id='67dbc289-8881-458f-825e-bc6aa9d8cb70', metadata={'date': '1974-11-20', 'event': 'Infiltration and Discovery of NK Tunnel System', 'description': 'During a joint U.S.-ROK investigation of a North Korean-dug infiltration tunnel, a North Korean explosive device sets off killing one American (USN Commander Robert M.

### 모델 생성

In [28]:
from langchain.chat_models import init_chat_model

model = init_chat_model('gpt-4.1-mini')

In [ ]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

def format_docs(docs):
  return '\n\n'.join([doc.page_content for doc in docs])


chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | model
    | StrOutputParser()
)